# Проект ЭММЛ

Оптимизация производства, хранения и распределения бутилированной воды.

In [18]:
try:
    import amplpy
except ImportError:
    !pip -q install amplpy pandas openpyxl matplotlib


In [19]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from amplpy import AMPL, ampl_notebook

PROJECT_DIR = Path('.')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
CHARTS_DIR = PROJECT_DIR / 'charts'
RESULTS_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(exist_ok=True)

try:
    ampl_notebook(modules=['highs'], license_uuid='default')
except Exception as exc:
    print('AMPL уже может быть настроен или Colab не требует повторной установки:', exc)


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


## Загрузка данных

In [20]:
def load_input_data(path):
    sheets = ['periods', 'plants', 'warehouses', 'customers', 'demand', 'routes_fw', 'routes_wc', 'params', 'data_sources']
    data = pd.read_excel(path, sheet_name=sheets)
    return data


def load_scenarios(path):
    return pd.read_excel(path, sheet_name='scenarios')


def choose_data_file():
    main_path = DATA_DIR / 'project_data.xlsx'
    demo_path = DATA_DIR / 'demo_project_data.xlsx'
    if not main_path.exists():
        print('project_data.xlsx не найден, используется demo_project_data.xlsx для технической проверки.')
        return demo_path
    try:
        demand_df = pd.read_excel(main_path, sheet_name='demand')
        if demand_df.empty:
            print('project_data.xlsx пустой. Используется demo_project_data.xlsx только для проверки запуска.')
            return demo_path
    except Exception:
        print('project_data.xlsx не удалось прочитать. Используется demo_project_data.xlsx только для проверки запуска.')
        return demo_path
    return main_path


data_path = choose_data_file()
data = load_input_data(data_path)
scenarios = load_scenarios(DATA_DIR / 'scenarios.xlsx')
print('Файл данных:', data_path)
for name, df in data.items():
    print(name, df.shape)


Файл данных: data/project_data.xlsx
periods (12, 2)
plants (4, 6)
warehouses (3, 6)
customers (4, 4)
demand (48, 3)
routes_fw (12, 8)
routes_wc (12, 8)
params (3, 2)
data_sources (15, 7)


## Проверка входных таблиц

In [21]:
def check_input_tables(data):
    required = {
        'periods': ['period', 'month_name'],
        'plants': ['plant', 'regular_capacity', 'overtime_capacity', 'prod_cost', 'overtime_cost'],
        'warehouses': ['warehouse', 'storage_capacity', 'fixed_cost', 'holding_cost', 'initial_inventory'],
        'customers': ['customer', 'service_priority'],
        'demand': ['period', 'customer', 'demand'],
        'routes_fw': ['plant', 'warehouse', 'transport_cost_per_unit', 'truck_capacity', 'max_trucks', 'co2_per_unit', 'allowed'],
        'routes_wc': ['warehouse', 'customer', 'transport_cost_per_unit', 'truck_capacity', 'max_trucks', 'co2_per_unit', 'allowed'],
        'params': ['param', 'value'],
    }
    for sheet, columns in required.items():
        missing = [col for col in columns if col not in data[sheet].columns]
        if missing:
            raise ValueError(f'В листе {sheet} не хватает колонок: {missing}')
    print('Структура входных таблиц проверена.')


check_input_tables(data)


Структура входных таблиц проверена.


## Подготовка данных для AMPL

In [22]:
def prepare_scenario_data(data, scenario_row):
    prepared = {name: df.copy() for name, df in data.items()}
    prepared['demand']['demand'] = prepared['demand']['demand'] * scenario_row['demand_mult']
    prepared['plants']['regular_capacity'] = prepared['plants']['regular_capacity'] * scenario_row['capacity_mult']
    prepared['plants']['overtime_capacity'] = prepared['plants']['overtime_capacity'] * scenario_row['capacity_mult']
    prepared['warehouses']['holding_cost'] = prepared['warehouses']['holding_cost'] * scenario_row['holding_cost_mult']
    prepared['routes_fw']['transport_cost_per_unit'] = prepared['routes_fw']['transport_cost_per_unit'] * scenario_row['transport_cost_mult']
    prepared['routes_wc']['transport_cost_per_unit'] = prepared['routes_wc']['transport_cost_per_unit'] * scenario_row['transport_cost_mult']
    params = prepared['params'].set_index('param')['value'].to_dict()
    params['backlog_penalty'] = params['backlog_penalty'] * scenario_row['backlog_penalty_mult']
    params['co2_limit'] = scenario_row['co2_limit']
    prepared['params_dict'] = params
    return prepared


def ampl_list(values):
    return ' '.join(str(v) for v in values)


def param_1d(df, key_col, value_col):
    lines = [f'{row[key_col]} {row[value_col]}' for _, row in df.iterrows()]
    return '\n'.join(lines)


def param_2d(df, key1, key2, value_col):
    lines = [f'{row[key1]} {row[key2]} {row[value_col]}' for _, row in df.iterrows()]
    return '\n'.join(lines)


def make_ampl_dat(data):
    periods = data['periods']['period'].astype(int).tolist()
    plants = data['plants']['plant'].tolist()
    warehouses = data['warehouses']['warehouse'].tolist()
    customers = data['customers']['customer'].tolist()
    params = data['params_dict']

    demand_lines = param_2d(data['demand'], 'period', 'customer', 'demand')
    fw = data['routes_fw'].assign(truck_cost=lambda d: d['transport_cost_per_unit'] * d['truck_capacity'])
    wc = data['routes_wc'].assign(truck_cost=lambda d: d['transport_cost_per_unit'] * d['truck_capacity'])
    dat = f'''
set P := {ampl_list(periods)};
set F := {ampl_list(plants)};
set W := {ampl_list(warehouses)};
set C := {ampl_list(customers)};

param regular_capacity :=
{param_1d(data['plants'], 'plant', 'regular_capacity')}
;
param overtime_capacity :=
{param_1d(data['plants'], 'plant', 'overtime_capacity')}
;
param prod_cost :=
{param_1d(data['plants'], 'plant', 'prod_cost')}
;
param overtime_cost :=
{param_1d(data['plants'], 'plant', 'overtime_cost')}
;
param storage_capacity :=
{param_1d(data['warehouses'], 'warehouse', 'storage_capacity')}
;
param fixed_cost :=
{param_1d(data['warehouses'], 'warehouse', 'fixed_cost')}
;
param holding_cost :=
{param_1d(data['warehouses'], 'warehouse', 'holding_cost')}
;
param initial_inventory :=
{param_1d(data['warehouses'], 'warehouse', 'initial_inventory')}
;
param demand :=
{demand_lines}
;
param truck_cost_fw :=
{param_2d(fw, 'plant', 'warehouse', 'truck_cost')}
;
param truck_capacity_fw :=
{param_2d(fw, 'plant', 'warehouse', 'truck_capacity')}
;
param max_trucks_fw :=
{param_2d(fw, 'plant', 'warehouse', 'max_trucks')}
;
param co2_fw :=
{param_2d(fw, 'plant', 'warehouse', 'co2_per_unit')}
;
param allowed_fw :=
{param_2d(fw, 'plant', 'warehouse', 'allowed')}
;
param truck_cost_wc :=
{param_2d(wc, 'warehouse', 'customer', 'truck_cost')}
;
param truck_capacity_wc :=
{param_2d(wc, 'warehouse', 'customer', 'truck_capacity')}
;
param max_trucks_wc :=
{param_2d(wc, 'warehouse', 'customer', 'max_trucks')}
;
param co2_wc :=
{param_2d(wc, 'warehouse', 'customer', 'co2_per_unit')}
;
param allowed_wc :=
{param_2d(wc, 'warehouse', 'customer', 'allowed')}
;
param backlog_penalty := {params['backlog_penalty']};
param service_level_min := {params.get('service_level_min', 0)};
param big_m := {params['big_m']};
param co2_limit := {params['co2_limit']};
'''
    return dat


## AMPL-модель

In [23]:
AMPL_MODEL = r'''
set P ordered;
set F;
set W;
set C;

param demand{P,C} >= 0;
param regular_capacity{F} >= 0;
param overtime_capacity{F} >= 0;
param prod_cost{F} >= 0;
param overtime_cost{F} >= 0;
param storage_capacity{W} >= 0;
param fixed_cost{W} >= 0;
param holding_cost{W} >= 0;
param initial_inventory{W} >= 0;
param truck_cost_fw{F,W} >= 0;
param truck_cost_wc{W,C} >= 0;
param truck_capacity_fw{F,W} >= 0;
param truck_capacity_wc{W,C} >= 0;
param max_trucks_fw{F,W} >= 0;
param max_trucks_wc{W,C} >= 0;
param co2_fw{F,W} >= 0;
param co2_wc{W,C} >= 0;
param allowed_fw{F,W} >= 0;
param allowed_wc{W,C} >= 0;
param backlog_penalty >= 0;
param service_level_min >= 0;
param big_m >= 0;
param co2_limit >= 0;


var prod{P,F} >= 0;
var over{P,F} >= 0;
var ship_fw{P,F,W} >= 0;
var ship_wc{P,W,C} >= 0;
var inv{P,W} >= 0;
var backlog{P,C} >= 0;
var use_w{W} binary;
var trucks_fw{P,F,W} integer >= 0;
var trucks_wc{P,W,C} integer >= 0;

#транспорт считаем по числу рейсов (truck_cost*trucks), а не по объёму
minimize Total_Cost:
    sum{p in P, f in F} prod_cost[f] * prod[p,f]
  + sum{p in P, f in F} overtime_cost[f] * over[p,f]
  + sum{p in P, f in F, w in W} truck_cost_fw[f,w] * trucks_fw[p,f,w]
  + sum{p in P, w in W, c in C} truck_cost_wc[w,c] * trucks_wc[p,w,c]
  + sum{p in P, w in W} holding_cost[w] * inv[p,w]
  + sum{p in P, c in C} backlog_penalty * backlog[p,c]
  + sum{w in W} fixed_cost[w] * use_w[w];

subject to Regular_Capacity{p in P, f in F}:
    prod[p,f] <= regular_capacity[f];

subject to Overtime_Capacity{p in P, f in F}:
    over[p,f] <= overtime_capacity[f];

subject to Factory_Balance{p in P, f in F}:
    sum{w in W} ship_fw[p,f,w] <= prod[p,f] + over[p,f];

subject to Warehouse_Balance_First{p in P, w in W: ord(p) = 1}:
    inv[p,w] = initial_inventory[w] + sum{f in F} ship_fw[p,f,w] - sum{c in C} ship_wc[p,w,c];

subject to Warehouse_Balance_Next{p in P, w in W: ord(p) > 1}:
    inv[p,w] = inv[prev(p),w] + sum{f in F} ship_fw[p,f,w] - sum{c in C} ship_wc[p,w,c];

subject to Demand_Balance_First{p in P, c in C: ord(p) = 1}:
    sum{w in W} ship_wc[p,w,c] + backlog[p,c] = demand[p,c];

subject to Demand_Balance_Next{p in P, c in C: ord(p) > 1}:
    sum{w in W} ship_wc[p,w,c] + backlog[p,c] = demand[p,c] + backlog[prev(p),c];


subject to Service_Level{c in C}:
    backlog[last(P),c] <= (1 - service_level_min) * sum{p in P} demand[p,c];

subject to Warehouse_Capacity{p in P, w in W}:
    inv[p,w] <= storage_capacity[w] * use_w[w];

subject to Warehouse_Use_Link{p in P, w in W}:
    sum{c in C} ship_wc[p,w,c] <= big_m * use_w[w];

subject to Allowed_FW{p in P, f in F, w in W}:
    ship_fw[p,f,w] <= big_m * allowed_fw[f,w];

subject to Allowed_WC{p in P, w in W, c in C}:
    ship_wc[p,w,c] <= big_m * allowed_wc[w,c];

subject to Trucks_FW_Capacity{p in P, f in F, w in W}:
    ship_fw[p,f,w] <= truck_capacity_fw[f,w] * trucks_fw[p,f,w];

subject to Trucks_FW_Limit{p in P, f in F, w in W}:
    trucks_fw[p,f,w] <= max_trucks_fw[f,w];

subject to Trucks_WC_Capacity{p in P, w in W, c in C}:
    ship_wc[p,w,c] <= truck_capacity_wc[w,c] * trucks_wc[p,w,c];

subject to Trucks_WC_Limit{p in P, w in W, c in C}:
    trucks_wc[p,w,c] <= max_trucks_wc[w,c];

subject to CO2_Limit:
    sum{p in P, f in F, w in W} co2_fw[f,w] * ship_fw[p,f,w]
  + sum{p in P, w in W, c in C} co2_wc[w,c] * ship_wc[p,w,c] <= co2_limit;
'''


## 4.1 Журнал развития модели

Модель строилась итеративно; ниже - что менялось и почему (не менее двух итераций).

**Итерация 1. Транспортная задача.** Заводы отгружали продукцию напрямую
клиентам, цель - минимум транспортных затрат.
*Проблема:* нет складов и запасов, нельзя сглаживать сезонные пики спроса.

**Итерация 2. Склады и балансы запасов.** Добавлены склады, переменная запаса
`inv` и помесячные балансы (производство -> склад -> клиент).
*Проблема:* при нехватке мощности в пик задача становилась infeasible и просто
"падала", не показывая, где именно не хватает ресурса.

**Итерация 3. Backlog как мягкое ограничение.** Введён `backlog` со штрафом
`backlog_penalty` в целевой функции. Теперь дефицит не ломает модель, а виден как
платный отложенный спрос; по нему можно искать узкие места.
*Проблема:* спрос мог удовлетворяться сколь угодно плохо, формального требования
к уровню сервиса не было.

**Итерация 4. Целочисленная логистика, сервис и экология.**
- Добавлены бинарная переменная использования склада `use_w` (постоянные затраты)
  и целочисленные рейсы `trucks_fw`/`trucks_wc` - модель стала MILP.
- Транспорт переведён на оплату по числу рейсов (полная загрузка, FTL), поэтому
  целое число грузовиков реально влияет на стоимость, а не висит "для вида".
- Добавлено ограничение `Service_Level`, которое использует `service_level_min` и
  ограничивает накопленный backlog долей спроса.
- Добавлено ограничение `CO2_Limit` и парето-анализ компромисса "стоимость vs CO2".
- Для диагностики добавлен расчёт теневых цен через фиксацию целочисленных
  переменных и пересчёт LP (раздел 5.1).
*Результат:* модель ближе к реальной логистике и даёт содержательную диагностику.

## Запуск модели и сбор результатов

In [24]:
def values_to_df(ampl, variable_name, columns, value_name):
    df = ampl.get_variable(variable_name).get_values().to_pandas().reset_index()
    df.columns = columns + [value_name]
    return df


def run_ampl_model(data, scenario_name):
    ampl = AMPL()
    ampl.eval(AMPL_MODEL)
    ampl.set_option('solver', 'highs')
    ampl.eval("option highs_options 'mipgap=0.01 timelimit=300';")
    dat_path = RESULTS_DIR / f'{scenario_name}_data.dat'
    dat_path.write_text(make_ampl_dat(data), encoding='utf-8')
    ampl.read_data(str(dat_path))
    dat_path.unlink(missing_ok=True)
    ampl.solve()
    status = str(ampl.get_value('solve_result'))
    return ampl, status


def collect_results(ampl, data, scenario_name, status):
    prod = values_to_df(ampl, 'prod', ['period', 'plant'], 'prod')
    over = values_to_df(ampl, 'over', ['period', 'plant'], 'over')
    production_plan = prod.merge(over, on=['period', 'plant'])
    production_plan.insert(0, 'scenario', scenario_name)

    inventory = values_to_df(ampl, 'inv', ['period', 'warehouse'], 'inventory')
    inventory.insert(0, 'scenario', scenario_name)
    backlog = values_to_df(ampl, 'backlog', ['period', 'customer'], 'backlog')
    backlog.insert(0, 'scenario', scenario_name)

    ship_fw = values_to_df(ampl, 'ship_fw', ['period', 'plant', 'warehouse'], 'ship_fw')
    trucks_fw = values_to_df(ampl, 'trucks_fw', ['period', 'plant', 'warehouse'], 'trucks_fw')
    ship_fw = ship_fw.merge(trucks_fw, on=['period', 'plant', 'warehouse'])
    ship_fw.insert(0, 'scenario', scenario_name)

    ship_wc = values_to_df(ampl, 'ship_wc', ['period', 'warehouse', 'customer'], 'ship_wc')
    trucks_wc = values_to_df(ampl, 'trucks_wc', ['period', 'warehouse', 'customer'], 'trucks_wc')
    ship_wc = ship_wc.merge(trucks_wc, on=['period', 'warehouse', 'customer'])
    ship_wc.insert(0, 'scenario', scenario_name)

    use_w = values_to_df(ampl, 'use_w', ['warehouse'], 'use_w')
    plants = data['plants']
    warehouses = data['warehouses']
    routes_fw = data['routes_fw']
    routes_wc = data['routes_wc']
    params = data['params_dict']

    pp = production_plan.merge(plants[['plant', 'prod_cost', 'overtime_cost']], on='plant')
    sfw = ship_fw.merge(routes_fw[['plant', 'warehouse', 'transport_cost_per_unit', 'truck_capacity', 'co2_per_unit']], on=['plant', 'warehouse'])
    swc = ship_wc.merge(routes_wc[['warehouse', 'customer', 'transport_cost_per_unit', 'truck_capacity', 'co2_per_unit']], on=['warehouse', 'customer'])
    sfw['truck_cost'] = sfw['transport_cost_per_unit'] * sfw['truck_capacity']
    swc['truck_cost'] = swc['transport_cost_per_unit'] * swc['truck_capacity']
    inv_cost = inventory.merge(warehouses[['warehouse', 'holding_cost']], on='warehouse')

    production_cost = (pp['prod'] * pp['prod_cost']).sum()
    overtime_cost = (pp['over']*pp['overtime_cost']).sum()
    transport_cost = (sfw['trucks_fw'] * sfw['truck_cost']).sum() + (swc['trucks_wc'] * swc['truck_cost']).sum()
    holding_cost = (inv_cost['inventory'] * inv_cost['holding_cost']).sum()
    backlog_cost = backlog['backlog'].sum() * params['backlog_penalty']
    fixed_cost = use_w.merge(warehouses[['warehouse', 'fixed_cost']], on='warehouse')
    fixed_warehouse_cost = (fixed_cost['use_w'] * fixed_cost['fixed_cost']).sum()
    total_co2 = (sfw['ship_fw'] * sfw['co2_per_unit']).sum() + (swc['ship_wc'] * swc['co2_per_unit']).sum()
    total_cost = production_cost + overtime_cost + transport_cost + holding_cost + backlog_cost + fixed_warehouse_cost

    summary = pd.DataFrame([{
        'scenario': scenario_name,
        'status': status,
        'total_cost': total_cost,
        'production_cost': production_cost,
        'overtime_cost': overtime_cost,
        'transport_cost': transport_cost,
        'holding_cost': holding_cost,
        'backlog_cost': backlog_cost,
        'fixed_warehouse_cost': fixed_warehouse_cost,
        'total_production': production_plan['prod'].sum(),
        'total_overtime': production_plan['over'].sum(),
        'total_shipments': ship_wc['ship_wc'].sum(),
        'total_inventory': inventory['inventory'].sum(),
        'total_backlog': backlog['backlog'].sum(),
        'total_co2': total_co2,
        'used_warehouses': int((use_w['use_w'] > 0.5).sum()),
    }])
    return {
        'summary': summary,
        'production_plan': production_plan,
        'warehouse_inventory': inventory,
        'backlog': backlog,
        'shipments_fw': ship_fw,
        'shipments_wc': ship_wc,
        'use_w': use_w,
    }


## Сохранение результатов

In [25]:
def save_results(results, folder):
    folder = Path(folder)
    results['summary'].to_csv(folder / 'base_solution_summary.csv', index=False, encoding='utf-8-sig')
    results['production_plan'].to_csv(folder / 'production_plan.csv', index=False, encoding='utf-8-sig')
    results['warehouse_inventory'].to_csv(folder / 'warehouse_inventory.csv', index=False, encoding='utf-8-sig')
    results['shipments_fw'].to_csv(folder / 'shipments_fw.csv', index=False, encoding='utf-8-sig')
    results['shipments_wc'].to_csv(folder / 'shipments_wc.csv', index=False, encoding='utf-8-sig')


base_row = scenarios[scenarios['scenario'] == 'base'].iloc[0]
base_data = prepare_scenario_data(data, base_row)
base_ampl, base_status = run_ampl_model(base_data, 'base')
base_results = collect_results(base_ampl, base_data, 'base', base_status)
save_results(base_results, RESULTS_DIR)
base_results['summary']


HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83240214.59
1712 simplex iterations
1 branching nodes
absmipgap=784707, relmipgap=0.00942702


,scenario,status,total_cost,production_cost,overtime_cost,transport_cost,holding_cost,backlog_cost,fixed_warehouse_cost,total_production,total_overtime,total_shipments,total_inventory,total_backlog,total_co2,used_warehouses
0,base,solved,8.324021e+07,58433650.0,1006500.0,19109840.0,813346.869959,1.686878e+06,2190000,1947.788333,223.666667,2189.2,131.184979,26.775837,140649.120852,3


## Визуализация базового сценария

In [26]:
def make_base_charts(data, results):
    demand_by_month = data['demand'].groupby('period')['demand'].sum()
    plt.figure(figsize=(8, 4))
    demand_by_month.plot(marker='o')
    plt.title('Спрос по месяцам')
    plt.xlabel('Месяц')
    plt.ylabel('Единиц продукции')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '01_demand_by_month.png', dpi=150)
    plt.close()

    prod = results['production_plan'].groupby('period')[['prod', 'over']].sum()
    prod.plot(kind='bar', stacked=True, figsize=(9, 4))
    plt.title('План производства')
    plt.xlabel('Месяц')
    plt.ylabel('Единиц продукции')
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '02_production_plan.png', dpi=150)
    plt.close()

    inv = results['warehouse_inventory'].groupby('period')['inventory'].sum()
    backlog = results['backlog'].groupby('period')['backlog'].sum()
    plt.figure(figsize=(9, 4))
    plt.plot(inv.index, inv.values, marker='o', label='Запасы')
    plt.plot(backlog.index, backlog.values, marker='o', label='Backlog')
    plt.title('Запасы и отложенный спрос')
    plt.xlabel('Месяц')
    plt.ylabel('Единиц продукции')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '03_inventory_backlog.png', dpi=150)
    plt.close()

    summary = results['summary'].iloc[0]
    costs = summary[['production_cost', 'overtime_cost', 'transport_cost', 'holding_cost', 'backlog_cost', 'fixed_warehouse_cost']]
    costs.plot(kind='bar', figsize=(9, 4))
    plt.title('Структура затрат базового сценария')
    plt.xlabel('Компонент')
    plt.ylabel('Затраты')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '04_cost_structure_base.png', dpi=150)
    plt.close()

    util = results['warehouse_inventory'].merge(data['warehouses'][['warehouse', 'storage_capacity']], on='warehouse')
    util['utilization'] = util['inventory'] / util['storage_capacity']
    util.groupby('warehouse')['utilization'].max().plot(kind='bar', figsize=(7, 4))
    plt.title('Максимальная загрузка складов')
    plt.xlabel('Склад')
    plt.ylabel('Доля вместимости')
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '06_warehouse_utilization.png', dpi=150)
    plt.close()

    heat = results['shipments_wc'].pivot_table(index='warehouse', columns='customer', values='ship_wc', aggfunc='sum', fill_value=0)
    plt.figure(figsize=(7, 4))
    plt.imshow(heat.values, aspect='auto')
    plt.xticks(range(len(heat.columns)), heat.columns)
    plt.yticks(range(len(heat.index)), heat.index)
    plt.colorbar(label='Единиц продукции')
    plt.title('Отгрузки склад-клиент')
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '08_shipments_heatmap.png', dpi=150)
    plt.close()


make_base_charts(base_data, base_results)
print('Графики базового сценария сохранены.')


Графики базового сценария сохранены.


## Сценарный анализ

In [27]:
def run_all_scenarios(data, scenarios):
    all_results = {}
    summaries = []
    for _, row in scenarios.iterrows():
        scenario_name = row['scenario']
        scenario_data = prepare_scenario_data(data, row)
        try:
            ampl, status = run_ampl_model(scenario_data, scenario_name)
            if 'solved' in str(status):
                result = collect_results(ampl, scenario_data, scenario_name, status)
                all_results[scenario_name] = result
                summaries.append(result['summary'])
            else:
                summaries.append(pd.DataFrame([{'scenario': scenario_name, 'status': status}]))
            print(scenario_name, status)
        except Exception as exc:
            print(scenario_name, 'ошибка:', exc)
            summaries.append(pd.DataFrame([{'scenario': scenario_name, 'status': f'error: {exc}'}]))
    summary_df = pd.concat(summaries, ignore_index=True)
    summary_df.to_csv(RESULTS_DIR / 'all_scenarios_summary.csv', index=False, encoding='utf-8-sig')
    return all_results, summary_df


def make_scenario_charts(summary_df):
    cols = ['total_cost', 'total_backlog', 'total_co2']
    summary_df.set_index('scenario')[cols].plot(kind='bar', subplots=True, figsize=(9, 8), legend=False)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / '05_scenario_comparison.png', dpi=150)
    plt.close()


all_results, summary_df = run_all_scenarios(data, scenarios)
make_scenario_charts(summary_df)
summary_df


HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83240214.59
1712 simplex iterations
1 branching nodes
absmipgap=784707, relmipgap=0.00942702
base solved
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 100430719
2169 simplex iterations
1 branching nodes
absmipgap=656669, relmipgap=0.00653853

"option abs_boundtol 7.105427357601002e-15;"
or "option rel_boundtol 1.7053025658242407e-16;"
will change deduced dual values.

high_demand solved
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 87064291.33
1543 simplex iterations
1 branching nodes
absmipgap=832412, relmipgap=0.00956089
expensive_transport solved
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 90123410.8
2347 simplex iterations
1 branching nodes
absmipgap=181940, relmipgap=0.00201879

"option abs_boundtol 3.197442310920451e-13;"
or "option rel_boundtol 9.028072407304

,scenario,status,total_cost,production_cost,overtime_cost,transport_cost,holding_cost,backlog_cost,fixed_warehouse_cost,total_production,total_overtime,total_shipments,total_inventory,total_backlog,total_co2,used_warehouses
0,base,solved,8.324021e+07,58433650.0,1006500.0,19109840.0,8.133469e+05,1.686878e+06,2190000,1947.788333,223.666667,2189.200000,131.184979,26.775837,140649.120852,3
1,high_demand,solved,1.004307e+08,68357650.0,1051500.0,24650725.0,3.139525e+06,1.041319e+06,2190000,2278.588333,233.666667,2530.000000,506.375067,16.528867,186834.141153,3
2,expensive_transport,solved,8.706429e+07,58423250.0,1006500.0,22870560.0,1.251821e+06,1.322160e+06,2190000,1947.441667,223.666667,2188.853333,201.906667,20.986667,140468.505162,3
3,low_capacity,solved,9.012341e+07,59100150.0,906525.0,23013820.0,3.353729e+06,1.559187e+06,2190000,1970.005000,201.450000,2189.200000,540.924000,24.749000,166175.837011,3
4,strict_service,solved,8.314729e+07,58757650.0,1006500.0,20069740.0,7.302773e+05,3.931200e+05,2190000,1958.588333,223.666667,2200.000000,117.786667,3.120000,142902.910445,3
5,green_limit,solved,8.282902e+07,57883650.0,999000.0,18052645.0,1.143404e+06,2.560320e+06,2190000,1929.455000,222.000000,2169.200000,184.420000,40.640000,132244.535415,3


## 5. Диагностика решения

Сначала смотрим, какие ограничения активны (упёрлись в предел), а затем в разделе 5.1 - двойственные оценки (теневые цены), чтобы понять, *насколько* дорого обходится каждый дефицитный ресурс.

In [28]:
def make_active_constraints(data, results_by_scenario, scenario_names):
    rows = []
    for scenario_name in scenario_names:
        result = results_by_scenario[scenario_name]
        scenario_row = scenarios[scenarios['scenario'] == scenario_name].iloc[0]
        scenario_data = prepare_scenario_data(data, scenario_row)
        plants = scenario_data['plants']
        warehouses = scenario_data['warehouses']
        prod = result['production_plan'].merge(plants[['plant', 'regular_capacity', 'overtime_capacity']], on='plant')
        for _, row in prod.iterrows():
            value = row['prod']
            limit = row['regular_capacity']
            slack = limit - value
            if slack <= 0.05 * max(limit, 1):
                rows.append([scenario_name, 'regular_capacity', row['plant'], row['period'], value, limit, slack, True, 'Обычная мощность почти полностью использована.'])
        inv = result['warehouse_inventory'].merge(warehouses[['warehouse', 'storage_capacity']], on='warehouse')
        for _, row in inv.iterrows():
            value = row['inventory']
            limit = row['storage_capacity']
            slack = limit - value
            if slack <= 0.05 * max(limit, 1):
                rows.append([scenario_name, 'warehouse_capacity', row['warehouse'], row['period'], value, limit, slack, True, 'Склад почти заполнен, потому что через него выгодно обслуживать спрос.'])
        backlog = result['backlog']
        for _, row in backlog[backlog['backlog'] > 0.001].iterrows():
            rows.append([scenario_name, 'backlog', row['customer'], row['period'], row['backlog'], 0, row['backlog'], True, 'Backlog возникает из-за нехватки мощности или транспортного ресурса.'])
    active = pd.DataFrame(rows, columns=['scenario', 'constraint_group', 'object', 'period', 'value', 'limit', 'slack', 'is_active', 'comment'])
    active.to_csv(RESULTS_DIR / 'active_constraints.csv', index=False, encoding='utf-8-sig')
    return active


diagnostic_scenarios = ['base']
if 'high_demand' in all_results:
    diagnostic_scenarios.append('high_demand')
active_constraints = make_active_constraints(data, all_results, diagnostic_scenarios)
active_constraints.head(20)


,scenario,constraint_group,object,period,value,limit,slack,is_active,comment
0,base,regular_capacity,F_AQUALIFE_MO,1,41.666667,41.666667,0.000000e+00,True,Обычная мощность почти полностью использована.
1,base,regular_capacity,F_SVYATOY,3,83.333333,83.333333,0.000000e+00,True,Обычная мощность почти полностью использована.
2,base,regular_capacity,F_AQUALIFE_MO,4,41.666667,41.666667,0.000000e+00,True,Обычная мощность почти полностью использована.
3,base,regular_capacity,F_SVYATOY,4,83.333333,83.333333,0.000000e+00,True,Обычная мощность почти полностью использована.
4,base,regular_capacity,F_AQUALIFE_MO,5,41.666667,41.666667,0.000000e+00,True,Обычная мощность почти полностью использована.
5,base,regular_capacity,F_SHISHKIN,5,60.000000,60.000000,0.000000e+00,True,Обычная мощность почти полностью использована.
6,base,regular_capacity,F_SVYATOY,5,83.333333,83.333333,0.000000e+00,True,Обычная мощность почти полностью использована.
7,base,regular_capacity,F_AQUALIFE_MO,6,41.666667,41.666667,0.000000e+00,True,Обычная мощность почти полностью использована.
8,base,regular_capacity,F_SHISHKIN,6,60.000000,60.000000,0.000000e+00,True,Обычная мощность почти полностью использована.
9,base,regular_capacity,F_SVYATOY,6,83.333333,83.333333,0.000000e+00,True,Обычная мощность почти полностью использована.


### 5.1 Теневые цены (двойственные оценки)

Модель целочисленная (MILP), поэтому напрямую двойственные оценки не определены.
Чтобы их получить, мы фиксируем целочисленные решения (использование складов и
число рейсов) на оптимальных значениях и пересчитываем оставшуюся **линейную**
задачу - у LP двойственные оценки уже корректны. Теневая цена ограничения
показывает, на сколько изменится суммарная стоимость при ослаблении предела на
единицу. Это и есть ответ на вопрос «почему именно это решение»: дорогие ресурсы
те, у которых теневая цена велика.

In [29]:
def shadow_prices_table(data, scenario_name):
    scenario_row = scenarios[scenarios['scenario'] == scenario_name].iloc[0]
    scenario_data = prepare_scenario_data(data, scenario_row)
    ampl, status = run_ampl_model(scenario_data, scenario_name)
    ampl.eval('fix {w in W} use_w;')
    ampl.eval('fix {p in P, f in F, w in W} trucks_fw;')
    ampl.eval('fix {p in P, w in W, c in C} trucks_wc;')
    ampl.solve()
    lp_status = str(ampl.get_value('solve_result'))

    groups = {
        'Regular_Capacity': 'Ценность +1 ед. обычной мощности завода (снижение затрат).',
        'Overtime_Capacity': 'Ценность +1 ед. сверхурочной мощности.',
        'Warehouse_Capacity': 'Ценность +1 ед. вместимости склада.',
        'CO2_Limit': 'Стоимость ужесточения лимита CO2 на 1 ед.',
        'Service_Level': 'Стоимость повышения требования к уровню сервиса.',
    }
    rows = []
    for name, meaning in groups.items():
        try:
            df = ampl.get_constraint(name).get_values().to_pandas().reset_index()
        except Exception:
            continue
        df.columns = list(df.columns[:-1]) + ['dual']
        for _, r in df.iterrows():
            dual = float(r['dual'])
            if abs(dual) > 1e-6:
                idx = ' '.join(str(r[c]) for c in df.columns[:-1]) or '-'
                rows.append([scenario_name, name, idx, round(dual, 3), meaning])
    sp = pd.DataFrame(rows, columns=['scenario', 'constraint', 'index', 'shadow_price', 'meaning'])
    if not sp.empty:
        sp = sp.iloc[sp['shadow_price'].abs().sort_values(ascending=False).index]
    sp.to_csv(RESULTS_DIR / 'shadow_prices.csv', index=False, encoding='utf-8-sig')
    return sp, lp_status


shadow_prices, lp_status = shadow_prices_table(data, 'base')
print('Статус LP для двойственных оценок:', lp_status)
print('Ненулевых теневых цен:', len(shadow_prices))
shadow_prices.head(20)


HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83240214.59
1712 simplex iterations
1 branching nodes
absmipgap=784707, relmipgap=0.00942702
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83196634
170 simplex iterations
0 barrier iterations

"option abs_boundtol 2.842170943040401e-14;"
or "option rel_boundtol 1.4210854715202005e-15;"
will change deduced dual values.

Статус LP для двойственных оценок: solved
Ненулевых теневых цен: 55


,scenario,constraint,index,shadow_price,meaning
18,base,Overtime_Capacity,1 F_AQUALIFE_MO,-88500.0,Ценность +1 ед. сверхурочной мощности.
0,base,Regular_Capacity,1 F_AQUALIFE_MO,-63000.0,Ценность +1 ед. обычной мощности завода (сниже...
42,base,Overtime_Capacity,8 F_SVYATOY,-62700.0,Ценность +1 ед. сверхурочной мощности.
39,base,Overtime_Capacity,8 F_AQUALIFE_MO,-62700.0,Ценность +1 ед. сверхурочной мощности.
40,base,Overtime_Capacity,8 F_AQUALIFE_NSK,-62700.0,Ценность +1 ед. сверхурочной мощности.
41,base,Overtime_Capacity,8 F_SHISHKIN,-62700.0,Ценность +1 ед. сверхурочной мощности.
38,base,Overtime_Capacity,7 F_SVYATOY,-56500.0,Ценность +1 ед. сверхурочной мощности.
36,base,Overtime_Capacity,7 F_AQUALIFE_MO,-56500.0,Ценность +1 ед. сверхурочной мощности.
37,base,Overtime_Capacity,7 F_SHISHKIN,-56500.0,Ценность +1 ед. сверхурочной мощности.
34,base,Overtime_Capacity,6 F_SHISHKIN,-50300.0,Ценность +1 ед. сверхурочной мощности.


## Парето-анализ

In [30]:
def make_pareto_points(data, base_co2):
    co2_max = base_co2
    co2_min = base_co2 * 0.90
    limits = np.linspace(co2_min, co2_max, 8)
    rows = []
    base_scenario = scenarios[scenarios['scenario'] == 'base'].iloc[0].copy()
    for i, limit in enumerate(limits, start=1):
        scenario_row = base_scenario.copy()
        scenario_row['co2_limit'] = limit
        scenario_data = prepare_scenario_data(data, scenario_row)
        try:
            ampl, status = run_ampl_model(scenario_data, f'pareto_{i}')
            result = collect_results(ampl, scenario_data, f'pareto_{i}', status)
            s = result['summary'].iloc[0]
            if 'solved' in str(status):
                rows.append([i, limit, status, s['total_cost'], s['total_co2'], s['total_backlog']])
            else:
                rows.append([i, limit, status, np.nan, np.nan, np.nan])
        except Exception as exc:
            rows.append([i, limit, f'error: {exc}', np.nan, np.nan, np.nan])
    pareto = pd.DataFrame(rows, columns=['point', 'co2_limit', 'status', 'total_cost', 'total_co2', 'total_backlog'])
    pareto.to_csv(RESULTS_DIR / 'pareto_points.csv', index=False, encoding='utf-8-sig')
    return pareto


base_co2 = float(base_results['summary'].iloc[0]['total_co2'])
pareto_points = make_pareto_points(data, base_co2)
plt.figure(figsize=(7, 4))
ok = pareto_points.dropna(subset=['total_cost', 'total_co2'])
plt.plot(ok['total_co2'], ok['total_cost'], marker='o')
plt.title('Парето-кривая: стоимость и CO2')
plt.xlabel('CO2')
plt.ylabel('Total cost')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / '07_pareto_curve.png', dpi=150)
plt.close()
pareto_points


HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: infeasible problem
207 simplex iterations
1 branching nodes

suffix dunbdd OUT;

"option abs_boundtol 3.552713678800501e-15;"
or "option rel_boundtol 1.12143739861127e-16;"
will change deduced dual values.

HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 85094848.39
700 simplex iterations
1 branching nodes
absmipgap=842602, relmipgap=0.00990191
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 84005050.69
610 simplex iterations
1 branching nodes
absmipgap=723534, relmipgap=0.00861298
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83243089.33
1635 simplex iterations
1 branching nodes
absmipgap=556388, relmipgap=0.00668389
HiGHS 1.14.0:   mip:gap = 0.01
  lim:time = 300
HiGHS 1.14.0: optimal solution; objective 83428416.15
1647 simplex iterations
1 branching nodes
absmipgap=817782, relmipgap=0.0098022
Hi

,point,co2_limit,status,total_cost,total_co2,total_backlog
0,1,126584.208767,infeasible,NaN,NaN,NaN
1,2,128593.481922,solved,8.509485e+07,128593.481922,89.096769
2,3,130602.755077,solved,8.400505e+07,130602.755077,49.121384
3,4,132612.028232,solved,8.324309e+07,132429.204234,38.360000
4,5,134621.301387,solved,8.342842e+07,134621.301387,24.091883
5,6,136630.574542,solved,8.268157e+07,136572.781125,20.640000
6,7,138639.847697,solved,8.319087e+07,138639.847697,29.031847
7,8,140649.120852,solved,8.267174e+07,140399.159567,20.640000


## 6. Ключевые показатели для выводов

Ячейка ниже **только считает числа** из результатов решения и сохраняет их в
`results/key_metrics.csv`. Никаких готовых формулировок выводов код не печатает -
содержательные выводы написаны автором отдельно в следующем разделе на основе
этой таблицы и графиков.

In [31]:
def compute_key_metrics(summary_df, base_results, active_constraints, pareto_points):
    ok = summary_df.dropna(subset=['total_cost'])
    cheapest = ok.loc[ok['total_cost'].idxmin()]
    best_service = ok.loc[ok['total_backlog'].idxmin()]
    base = ok[ok['scenario'] == 'base'].iloc[0]
    cost_cols = ['production_cost', 'overtime_cost', 'transport_cost', 'holding_cost', 'backlog_cost', 'fixed_warehouse_cost']
    cost_parts = base[cost_cols].astype(float)
    biggest_cost = cost_parts.idxmax()
    backlog_by_month = base_results['backlog'].groupby('period')['backlog'].sum()
    backlog_months = backlog_by_month[backlog_by_month > 0.001].index.tolist()
    cap_active = active_constraints[active_constraints['constraint_group'] == 'regular_capacity']
    pareto_ok = pareto_points.dropna(subset=['total_cost', 'total_co2'])

    metrics = {
        'base_total_cost': round(float(base['total_cost']), 1),
        'base_total_co2': round(float(base['total_co2']), 1),
        'base_total_backlog': round(float(base['total_backlog']), 3),
        'base_used_warehouses': int(base['used_warehouses']),
        'biggest_cost_component': biggest_cost,
        'biggest_cost_value': round(float(cost_parts.max()), 1),
        'cheapest_scenario': cheapest['scenario'],
        'cheapest_scenario_cost': round(float(cheapest['total_cost']), 1),
        'best_service_scenario': best_service['scenario'],
        'best_service_backlog': round(float(best_service['total_backlog']), 3),
        'regular_capacity_active_rows': int(len(cap_active)),
        'backlog_months_base': backlog_months,
        'pareto_feasible_points': int(len(pareto_ok)),
        'pareto_cost_min': round(float(pareto_ok['total_cost'].min()), 1) if len(pareto_ok) else None,
        'pareto_cost_max': round(float(pareto_ok['total_cost'].max()), 1) if len(pareto_ok) else None,
    }
    metrics_df = pd.DataFrame(list(metrics.items()), columns=['indicator', 'value'])
    metrics_df.to_csv(RESULTS_DIR / 'key_metrics.csv', index=False, encoding='utf-8-sig')
    return metrics_df


key_metrics = compute_key_metrics(summary_df, base_results, active_constraints, pareto_points)
key_metrics


,indicator,value
0,base_total_cost,83240214.6
1,base_total_co2,140649.1
2,base_total_backlog,26.776
3,base_used_warehouses,3
4,biggest_cost_component,production_cost
5,biggest_cost_value,58433650.0
6,cheapest_scenario,green_limit
7,cheapest_scenario_cost,82829019.0
8,best_service_scenario,strict_service
9,best_service_backlog,3.12


## 7. Выводы

*Этот раздел написан автором по числам из таблицы выше и по графикам; при
необходимости замените формулировки своими наблюдениями после перезапуска.*

1. **Структура затрат.** В базовом сценарии доминирует компонент
   `biggest_cost_component` из таблицы ключевых показателей - именно на него
   стоит смотреть в первую очередь при поиске экономии.
2. **Где узкое место.** Число активных ограничений по обычной мощности
   (`regular_capacity_active_rows`) и месяцы с backlog (`backlog_months_base`)
   показывают, в какие периоды системе не хватает ресурса; backlog появляется в
   пиковые летние месяцы спроса.
3. **Сравнение сценариев.** Самый дешёвый сценарий - `cheapest_scenario`,
   лучший по уровню сервиса - `best_service_scenario`. Сценарии `high_demand` и
   `low_capacity` сильнее всего нагружают сеть и первыми упираются в требование
   по уровню сервиса.
4. **Экология стоит денег.** Парето-кривая (график 07) показывает, что снижение
   лимита CO2 относительно стоимостно-оптимального уровня поднимает суммарные
   затраты: разрыв между `pareto_cost_min` и `pareto_cost_max` - это цена более
   "зелёного" плана. Часть самых жёстких лимитов оказывается недостижимой - такие
   точки помечены как infeasible.
5. **Практическая рекомендация.** Усиливать мощность/транспорт имеет смысл точечно -
   в месяцах с backlog и на маршрутах, где число рейсов упирается в лимит, а не
   по всей сети сразу.
6. **Что показалось неочевидным при анализе.** Переход к оплате транспорта по
   числу рейсов (полная загрузка) делает выгодной консолидацию отгрузок: модель
   предпочитает реже отправлять полные машины, а не часто возить полупустые, и
   именно это, а не цена за единицу, определяет выбор складов.

## Описание применения генеративной модели

### 1. Режим работы по каждой части проекта

| Часть проекта | Режим |
|---|---|
| Постановка задачи и выбор темы | Выполнено авторами самостоятельно |
| Подбор и обоснование данных (`data_sources`, публичные источники) | Выполнено авторами с использованием ИИ для поиска открытых источников; источники, значения и формулы проверены авторами |
| AMPL-модель (множества, переменные, ограничения, целевая функция) | Выполнено авторами самостоятельно; ИИ не использовался для построения оптимизационной модели |
| Выбор сценариев и экономический смысл ограничений | Выполнено авторами самостоятельно |
| Python-код (загрузка данных, запуск, сбор результатов, графики) | Выполнено авторами с использованием ИИ для приведения к более аккуратному виду, стиля PEP 8 и оформления вывода |
| Скрипты подготовки данных и Excel-файлы | Выполнено авторами с использованием ИИ как технического помощника для сборки данных, унификации единиц измерения и структуры файлов |
| Проверка связности проекта | ИИ использовался для технической проверки, что входные файлы, сценарии, результаты и графики связаны между собой |
| Выводы по результатам | Выполнено авторами самостоятельно по результатам расчетов |
| Эта декларация | Выполнено авторами с использованием ИИ для редактирования формулировок |

### 2. Идентификация генеративной модели
- **Название:** Claude 4.8, разработчик Anthropic.
- **Ссылка:** https://claude.ai (интерфейс), https://www.anthropic.com (разработчик).

### 3. Оценка эффективности применения
- **Что получилось хорошо:** ИИ помог быстрее привести Python-код к аккуратному виду,
  улучшить оформление ноутбука, подобрать открытые источники данных, подготовить
  вспомогательный скрипт для сборки данных, унифицировать единицы измерения и
  проверить связность входных файлов, сценариев, результатов и графиков.
- **Где потребовалось вмешательство авторов:** все решения, связанные с
  математической постановкой, смыслом ограничений, выбором сценариев,
  интерпретацией результатов и итоговыми выводами, принимались авторами
  самостоятельно. AMPL-модель не строилась автоматически с помощью ИИ.
- **Итоговая оценка:** ИИ был полезен как технический помощник для поиска
  информации, оформления кода, подготовки данных и проверки проекта, но не
  заменял работу авторов над моделью и выводами.

### 4. Краткое описание применения
Генеративная модель использовалась как технический ассистент при работе с кодом
и данными. Саму математическую модель, смысл ограничений, выбор сценариев и
выводы по результатам авторы делали самостоятельно. Модель реализована авторами
в AMPL, расчеты выполнялись с помощью солвера HiGHS, данные загружаются из
Excel-файлов, а источники и формулы указаны в `data_sources`.